# La3+ Metadynamics with MACE-Polar via Axon

Well-tempered metadynamics on La3+ coordination number with OH- and F- in 128-water
droplets, Langevin NVT at 300 K. Uses MACE-POLAR-1-S on L40S GPUs via Axon Modal sandboxes.

## Workflow
1. Upload equilibrated structures to `/mnt/remote`
2. Create `chemistry-py-3.12-gpu` sandboxes (L40S, 24h timeout)
3. Install: `mace-torch==0.3.16`, `graph_electrostatics==v0.4.0`
4. Launch `deep-shell/la_metadynamics.py` as subprocess (avoids kernel heartbeat timeout)
5. Monitor convergence via HILLS files on `/mnt/remote`
6. Download and analyze locally

## Key Parameters
- Model: MACE-POLAR-1-S (float32, CUDA), charge=0, spin=1
- Metadynamics: sigma=0.15, w0=2.0 kJ/mol, gamma=15, pace=500 steps
- CN switching function: n=6, m=12, R0=3.5 A (OH-) / 3.2 A (F-)
- Integrator: Langevin, 1.0 fs, 300 K, friction=0.01/fs, FixCom
- Checkpointing: pickle every 2000 steps (atomic write)

## Analysis

This notebook analyzes completed metadynamics runs. The simulation itself runs
via `deep-shell/la_metadynamics.py` inside Axon GPU sandboxes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path('../data/f-block-electrolytes')

SIGMA = 0.15
BIAS_FACTOR = 15
EV_TO_KJ = 96.485

SYSTEMS = {
    'oh': {'label': 'La3+ + 3 OH-', 'color': '#5CC2E1', 'r0': 3.5},
    'f':  {'label': 'La3+ + 3 F-',  'color': '#CB62BB', 'r0': 3.2},
}

def load_hills(path):
    data = np.loadtxt(path, comments='#')
    return data[:, 0], data[:, 1], data[:, 2]

def load_colvar(path):
    data = np.loadtxt(path, comments='#')
    return data[:, 0], data[:, 1], data[:, 2], data[:, 3], data[:, 4]

def reconstruct_fes(cn_hills, weight_hills, cn_grid):
    fes = np.zeros_like(cn_grid)
    for cn_k, w_k in zip(cn_hills, weight_hills):
        fes += w_k * np.exp(-(cn_grid - cn_k)**2 / (2 * SIGMA**2))
    fes *= -(BIAS_FACTOR / (BIAS_FACTOR - 1))
    fes -= fes.min()
    return fes * EV_TO_KJ

In [ ]:
cn_grid = np.linspace(-0.2, 3.5, 600)
results = {}

for tag, cfg in SYSTEMS.items():
    hills_path = DATA_DIR / f'hills_{tag}.dat'
    colvar_path = DATA_DIR / f'colvar_{tag}.dat'
    if not hills_path.exists():
        print(f'{tag}: no data'); continue
    steps, cn, w = load_hills(hills_path)
    csteps, ccn, cbias, ce, ct = load_colvar(colvar_path)
    fes = reconstruct_fes(cn, w, cn_grid)
    results[tag] = {'steps': steps, 'cn': cn, 'w': w, 'fes': fes,
                    'csteps': csteps, 'ccn': ccn, 'ct': ct, 'ce': ce}
    min_idx = np.argmin(fes)
    print(f"{cfg['label']}: {len(cn)} hills, {steps[-1]/1000:.0f} ps, "
          f"FES min at CN={cn_grid[min_idx]:.2f}, "
          f"T={np.mean(ct):.1f}+/-{np.std(ct):.1f} K")

### FES Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for tag, r in results.items():
    cfg = SYSTEMS[tag]
    n_ps = r['steps'][-1] / 1000
    ax.plot(cn_grid, r['fes'], color=cfg['color'], lw=2.2,
            label=f"{cfg['label']} ({n_ps:.0f} ps)")
ax.set_xlabel('Coordination Number')
ax.set_ylabel('Free Energy (kJ/mol)')
ax.set_title('FES: La3+ Anion Coordination (MACE-POLAR-1-S, 300 K)')
ax.set_xlim(-0.1, 3.3); ax.set_ylim(bottom=0)
ax.legend(frameon=False)
plt.show()

### CN Time Series

In [ ]:
fig, axes = plt.subplots(len(results), 1, figsize=(7, 3*len(results)), sharex=True)
if len(results) == 1: axes = [axes]
for ax, (tag, r) in zip(axes, results.items()):
    cfg = SYSTEMS[tag]
    ax.plot(r['csteps']/1000, r['ccn'], color=cfg['color'], lw=0.6, alpha=0.8)
    ax.set_ylabel(f"CN ({cfg['label']})")
    for y in [1, 2, 3]: ax.axhline(y=y, color='#E6E6E7', ls='--', lw=0.6)
    ax.set_ylim(-0.2, 3.5)
axes[-1].set_xlabel('Time (ps)')
axes[0].set_title('Coordination Number Time Series')
plt.show()

### FES Convergence

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5*len(results), 4), sharey=True)
if len(results) == 1: axes = [axes]
for ax, (tag, r) in zip(axes, results.items()):
    cfg = SYSTEMS[tag]
    n = len(r['cn'])
    for frac, alpha, ls, lab in [(0.25, 0.3, '--', '25%'),
                                  (0.50, 0.5, '-.', '50%'),
                                  (0.75, 0.7, ':', '75%'),
                                  (1.0, 1.0, '-', '100%')]:
        k = max(1, int(n * frac))
        fes = reconstruct_fes(r['cn'][:k], r['w'][:k], cn_grid)
        ax.plot(cn_grid, fes, color=cfg['color'], lw=1.6 if frac==1 else 1.0,
                alpha=alpha, ls=ls, label=f"{int(r['steps'][k-1]/1000)} ps")
    ax.set_title(f"{cfg['label']} Convergence")
    ax.set_xlabel('CN')
    ax.legend(frameon=False, fontsize=8)
axes[0].set_ylabel('Free Energy (kJ/mol)')
plt.show()

### Summary

In [ ]:
for tag, r in results.items():
    cfg = SYSTEMS[tag]
    fes = r['fes']
    min_idx = np.argmin(fes)
    print(f"--- {cfg['label']} ---")
    print(f"  Hills: {len(r['cn'])}, last step: {r['steps'][-1]:.0f} ({r['steps'][-1]/1000:.0f} ps)")
    print(f"  CN range: {r['ccn'].min():.3f} - {r['ccn'].max():.3f}")
    print(f"  FES minimum at CN = {cn_grid[min_idx]:.2f}")
    print(f"  Final hill weight: {r['w'][-1]*EV_TO_KJ:.3f} kJ/mol (decay: {r['w'][-1]/r['w'][0]:.3f})")
    print(f"  Temperature: {np.mean(r['ct']):.1f} +/- {np.std(r['ct']):.1f} K")
    print(f"  Total energy: {np.mean(r['ce']):.1f} +/- {np.std(r['ce']):.1f} eV")
    # Barrier
    mask = (cn_grid >= 1.0) & (cn_grid <= 2.5) if tag == 'oh' else (cn_grid >= 0.5) & (cn_grid <= 2.0)
    if np.any(mask):
        barrier = np.max(fes[mask]) - fes[min_idx]
        print(f"  Barrier: {barrier:.1f} kJ/mol")
    print()